# Catalogue des formations Zélia : audit et réparation compatible

Cette version remplace le traitement manuel en dix blocs du notebook historique. **Aucune cellule ne supprime une table ou ne réattribue un identifiant.** L'audit est en lecture seule ; la publication finale exige une sauvegarde complète, un plan validé et une activation explicite. L'original reste intact.

L'objectif n'est pas de remplir artificiellement tous les champs : chaque enrichissement doit être traçable, les inconnues doivent rester visibles et l'application doit continuer à fonctionner. Une donnée fiable améliore le test ; elle ne garantit pas, à elle seule, une orientation parfaite.

Le code testé se trouve dans `../scripts/formation_catalog.py`. Python 3.10+ et les dépendances Node déjà utilisées par le serveur suffisent ; aucun package Python supplémentaire n'est requis.

In [ ]:
import importlib.util
import json
import os
import sys
from pathlib import Path

candidates = [Path(os.environ['ZELIA_REPO'])] if os.environ.get('ZELIA_REPO') else [Path.cwd(), *Path.cwd().parents]
REPO = next((path for path in candidates if (path / 'server/scripts/formation_catalog.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Ouvrir le notebook depuis le dépôt Zélia ou définir ZELIA_REPO vers ce dépôt.')
spec = importlib.util.spec_from_file_location('formation_catalog', REPO / 'server/scripts/formation_catalog.py')
catalog = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = catalog
spec.loader.exec_module(catalog)
print('Module de contrôle chargé. Aucun accès à la base effectué.')

## 1. Contrat à préserver avec l'application

| Champ existant | Règle de compatibilité |
| --- | --- |
| `id` | Identifiant numérique stable ; utilisé par les fiches, liens, recommandations et sélections. Jamais recréé. |
| `nm`, `tf`, `fl`, `amg` | Listes de textes ou `null`, pas des chaînes à la place des tableaux. |
| `nmc` | Nom court provenant de la source. Ce n'est pas un identifiant unique ni une taxonomie universelle. |
| `tc` | Secteur/type d'établissement, pas type de diplôme. |
| `app`, `int`, `aut` | Textes conservés pour les consommateurs existants ; `int` n'est pas une preuve de niveau doctorat. |
| `annee`, `gta`, `gti`, `etab_uai` | Identifiants et session source conservés. Un portail `gti` et une offre `gta` ne sont pas interchangeables. |
| `commune`, `departement`, `geom` | Site réel d'enseignement ; ne pas substituer automatiquement le siège de l'université. |

L'analyse produit des pistes d'études et de métiers ; la recherche nationale rapproche ensuite ces pistes des vraies lignes de `formation_france`. Les listes, les identifiants et les libellés doivent donc rester compatibles avec les deux étapes.

## 2. Classification : diplôme, niveau et grade sont distincts

- BTS/BTSA/BTSM, DUT, DEUST ; BUT ; licence/licence professionnelle ; master ; DN MADE ; DCG/DSCG ; CAP/bac professionnel ; préparations et autres familles.
- `L1` décrit une première année de licence, pas un diplôme de niveau bac+1.
- Dans cette source, `DNM-Numérique` peut désigner un **DN MADE**, pas un master.
- **DU/DIU** : diplôme propre à une université, sans niveau ni grade automatique. **PhD** : doctorat de recherche. Un doctorat d'exercice en santé est encore une catégorie distincte.
- Bachelor, MBA, mastère et titre RNCP ne sont pas automatiquement des diplômes nationaux ni des grades de licence/master.
- Les CPGE et le PASS ne doivent pas être transformés en diplômes terminaux.
- Les cas contradictoires ou non reconnus sont marqués `needs_review`. Pas de LLM pour inventer les diplômes ou les intitulés.
- Une famille reconnue ne signifie pas que son niveau ou son grade sont établis : ces champs peuvent rester `None`. Le rapport mesure séparément leur couverture, notamment pour les certifications, FCIL et diplômes de santé.

Références : [niveaux officiels](https://www.service-public.gouv.fr/particuliers/vosdroits/F199), [champs Parcoursup](https://data.enseignementsup-recherche.gouv.fr/explore/dataset/fr-esr-cartographie_formations_parcoursup/). La classification proposée est un résultat d'audit séparé ; elle ne modifie pas les règles métier de l'application.

In [ ]:
examples = [
    {'nmc': 'BTS-SIO', 'nm': ['BTS Services informatiques aux organisations']},
    {'nmc': 'L1-Info', 'nm': ['Licence - Informatique']},
    {'nmc': 'DNM-Numérique', 'nm': ['DN MADE Numérique']},
    {'nmc': 'M2-Droit', 'nm': ['Master Droit']},
    {'nmc': 'DU-Nutrition', 'nm': ['DU Nutrition']},
    {'nmc': 'PhD', 'nm': ['PhD Computer Science']},
    {'nm': ["Diplôme d'État de docteur en médecine"]},
    {'nm': ['Mastère spécialisé cybersécurité']},
    {'nmc': 'CPGE-MPSI', 'nm': ['CPGE MPSI']},
]
for row in examples:
    result = catalog.classify_formation(row)
    print(row.get('nmc') or row['nm'][0], '=>', {key: result[key] for key in ['diploma_family', 'qualification_bac_level', 'qualification_rncp_level', 'academic_grade', 'status']})

## 3. Snapshot en lecture seule et comparaison avec la source

Le petit export Node réutilise `server/config/supabase.js` et l'environnement déjà utilisé par l'application. **Ne pas recopier une clé, un mot de passe ou une URL de connexion avec identifiants dans le notebook.** Le notebook historique en contenait : retirer ces secrets et renouveler ceux qui ont été exposés.

Commencer par 200 lignes. Un échantillon ne prouve pas la qualité de toute la base. Pour l'audit complet, activer explicitement `ALL_ROWS` après avoir examiné l'échantillon ; privilégier un export officiel local pour éviter de multiplier les appels réseau.

Le rapprochement de réparation utilise **session + gta + UAI**, pas seulement un nom ou un code de diplôme. Plusieurs correspondances, changement de nom court et valeurs existantes contradictoires sont signalés pour revue. Les sessions historiques ne sont pas fusionnées automatiquement.

In [ ]:
RUN_DATABASE_AUDIT = False
FETCH_OFFICIAL_MATCHES = False
INSPECT_COMPLEMENTARY_SOURCES = False
SAVE_PREVIEW = False
PUBLISH_VALIDATED_PLAN = False
PUBLICATION_PLAN = None
BACKUP_SNAPSHOT = None
PUBLICATION_JOURNAL = None
ALL_ROWS = False
SAMPLE_LIMIT = 200
OFFICIAL_EXPORT = None  # Optionnel : chemin d'un export JSON/GeoJSON officiel local.
snapshot = None
source_records = None
provenance = None
repair_plan = None
print('Lecture et publication Supabase désactivées par défaut. Activer les étapes une par une après validation.')

In [ ]:
if RUN_DATABASE_AUDIT:
    snapshot = catalog.database_snapshot(REPO / 'server', limit=SAMPLE_LIMIT, all_rows=ALL_ROWS)
    print(json.dumps(catalog.audit_rows(snapshot['rows'], complete=snapshot['complete']), ensure_ascii=False, indent=2))
    print('Lignes examinées / total base :', len(snapshot['rows']), '/', snapshot['total_rows'])
else:
    print('Étape non exécutée : RUN_DATABASE_AUDIT=False.')

In [ ]:
if FETCH_OFFICIAL_MATCHES:
    if snapshot is None:
        raise RuntimeError('Exécuter le snapshot en lecture seule avant de comparer la source.')
    if OFFICIAL_EXPORT:
        import hashlib
        content = Path(OFFICIAL_EXPORT).read_bytes()
        source_records = json.loads(content)
        if isinstance(source_records, dict) and source_records.get('type') == 'FeatureCollection':
            source_records = [feature['properties'] for feature in source_records['features']]
        if not isinstance(source_records, list):
            raise ValueError('Export officiel attendu : tableau JSON ou FeatureCollection GeoJSON.')
        provenance = {'dataset': catalog.DATASET, 'file': Path(OFFICIAL_EXPORT).name, 'sha256': hashlib.sha256(content).hexdigest()}
    else:
        source_records, provenance = catalog.fetch_matching_parcoursup(snapshot['rows'])
    repair_plan = catalog.build_repair_plan(snapshot['rows'], source_records, provenance=provenance, complete_snapshot=snapshot['complete'])
    print(json.dumps(repair_plan['summary'], ensure_ascii=False, indent=2))
    for change in repair_plan['changes'][:10]:
        print(json.dumps({'id': change['id'], 'avant': change['before'], 'après': change['after'], 'famille': change['classification']['diploma_family']}, ensure_ascii=False))
else:
    print('Étape non exécutée : FETCH_OFFICIAL_MATCHES=False.')

## 4. Compléter le périmètre : ne pas confondre les sources

Parcoursup n'est pas un catalogue complet de tous les diplômes du supérieur. [Mon Master](https://data.enseignementsup-recherche.gouv.fr/explore/dataset/fr-esr-mon_master/) contient des offres observées lors de campagnes annuelles : leur année doit rester explicite, et les campus, coordonnées et liens actuels doivent être vérifiés avant publication.

L'[annuaire des écoles doctorales](https://data.enseignementsup-recherche.gouv.fr/explore/dataset/fr-esr-ecoles_doctorales_annuaire/) décrit des structures et des accréditations, pas des places disponibles en doctorat. Une date d'accréditation ancienne nécessite une vérification ; ni l'annuaire ni une thèse soutenue ne prouvent qu'un recrutement est ouvert.

Les aperçus ci-dessous restent séparés de la réparation de l'existant. Un identifiant Mon Master ne doit jamais être copié dans `gta`, et une école doctorale ne doit pas devenir artificiellement une ligne de formation. Les partenaires sont hors périmètre.

In [ ]:
if INSPECT_COMPLEMENTARY_SOURCES:
    complementary = catalog.complementary_source_previews(limit=5)
    print(json.dumps(complementary, ensure_ascii=False, indent=2))
else:
    print('Inspection des sources complémentaires non exécutée.')

In [ ]:
if SAVE_PREVIEW:
    if repair_plan is None:
        raise RuntimeError('Aucune comparaison n’a été exécutée : il n’y a pas de plan à exporter.')
    from datetime import datetime, timezone
    suffix = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    target = REPO / 'server/.generated' / f'formation-catalog-preview-{suffix}.json'
    catalog.save_preview(repair_plan, target)
    print('Aperçu enregistré :', target)
else:
    print('Aucun fichier de données exporté. Aucune modification Supabase effectuée.')

## 5. Conditions avant toute publication

1. Audit **complet**, couverture mesurée par session et famille ; examiner les correspondances absentes et ambiguës.
2. Valider les propositions sur une table de préparation, jamais avec `DROP TABLE` sur la table utilisée par l'application.
3. Sauvegarder les valeurs initiales et conserver les `id`, liens, sélections et rattachements existants.
4. Appliquer uniquement les champs validés avec un contrôle des valeurs initiales : une modification concurrente impose une nouvelle revue.
5. Tester avant/après les recherches par diplôme, domaine et département ainsi que leurs temps de réponse ; préparer un retour arrière.
6. Séparer offre actuelle et archives. Conserver la date et la source de chaque mise à jour ; aucune garantie de disponibilité actuelle ne doit être déduite d'un ancien millésime.

Les métadonnées de classification restent dans le rapport : la publication ne remplit que les champs existants vérifiés dans la source officielle, sans modifier le schéma ni prétendre compléter les cas incertains.

Vérifications depuis la racine du dépôt :
```powershell
python -m unittest discover -s server\scripts -p test_formation_catalog.py -v
cd server
node --test scripts\verify_national_formation_matching.test.mjs scripts\verify_catalog_search.test.mjs
```

## 6. Publication contrôlée après approbation

Définir `BACKUP_SNAPSHOT`, `PUBLICATION_PLAN` et un nouveau `PUBLICATION_JOURNAL` vers les fichiers validés. La sauvegarde doit contenir toutes les lignes. Le plan est toujours validé en simulation avant une application.

Le publieur ne fait que des mises à jour conditionnelles de champs manquants : il vérifie les valeurs initiales et l'identité source, puis les valeurs enregistrées. Une erreur ou une modification concurrente arrête la publication. Le journal conserve les lots vérifiés ; une reprise vérifie également les lots déjà appliqués. La sauvegarde et le plan permettent un retour arrière avec les mêmes contrôles (`--rollback --apply`), sans écraser un changement ultérieur.

**Activer `PUBLISH_VALIDATED_PLAN=True` seulement après accord explicite. Aucun `DROP`, `DELETE`, `INSERT` ou remplacement d'identifiant n'est exécuté.** La publication est atomique par lot, pas pour l'ensemble du catalogue : conserver le journal et vérifier ensuite toutes les lignes.

In [ ]:
if PUBLISH_VALIDATED_PLAN:
    import subprocess
    if not all([BACKUP_SNAPSHOT, PUBLICATION_PLAN, PUBLICATION_JOURNAL]):
        raise ValueError('Sauvegarde complète, plan validé et nouveau journal sont obligatoires.')
    command = [
        'node', '--max-old-space-size=3072', str(REPO / 'server/scripts/publish_formation_repairs.mjs'),
        '--backup', str(BACKUP_SNAPSHOT), '--plan', str(PUBLICATION_PLAN),
        '--journal', str(PUBLICATION_JOURNAL), '--apply',
    ]
    subprocess.run(command, cwd=REPO / 'server', check=True)
else:
    print('Publication non exécutée : PUBLISH_VALIDATED_PLAN=False.')